# B3 — Python Exploratory Data Analysis

This notebook consumes the validated PostgreSQL dimensional warehouse created in B2.

**Analytical period:** 2017-01-01 through 2018-08-31  
**Primary population:** delivered orders  
**Primary grain:** one row per order unless explicitly stated otherwise

## Objectives

1. Build a safe order-level analytical dataset without fact-table fanout.
2. Reconcile Python results with the validated SQL/warehouse results.
3. Analyze marketplace growth and revenue concentration.
4. Quantify the association between delivery performance and customer satisfaction.
5. Describe customer geography and repeat purchasing.

The notebook preserves the warehouse's natural fact grains. Item, payment, and review facts are aggregated to order grain before they are merged with `fact_orders`.


## 1. Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from src.database import read_query

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

FIGURE_DIR = Path("../docs/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "font.size": 10,
})

def save_figure(filename: str) -> None:
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches="tight")
    plt.show()


## 2. Build the order-level analytical dataset

### 2.1 Orders

`dw.fact_orders` already has one row per order. Customer identity, transaction geography, and order status are resolved through conformed dimensions.


In [ ]:
orders = read_query("""
SELECT
    fo.order_id,
    dc.customer_unique_id,
    dl.city AS customer_city,
    dl.state AS customer_state,
    dos.order_status,
    fo.order_purchase_timestamp,
    fo.order_approved_at,
    fo.order_delivered_carrier_date,
    fo.order_delivered_customer_date,
    fo.order_estimated_delivery_date,
    fo.delivery_days,
    fo.delivery_delay_days,
    fo.is_late_delivery
FROM dw.fact_orders AS fo
JOIN dw.dim_customer AS dc
    ON fo.customer_key = dc.customer_key
LEFT JOIN dw.dim_location AS dl
    ON fo.customer_location_key = dl.location_key
JOIN dw.dim_order_status AS dos
    ON fo.order_status_key = dos.order_status_key;
""")

assert len(orders) == 99_441
assert orders["order_id"].nunique() == len(orders)
assert not orders["order_id"].duplicated().any()

print("Orders:", len(orders))
orders.head()


### 2.2 Order-item aggregates

`fact_order_items` has one row per `(order_id, order_item_id)`. It is aggregated before joining to the order table so that orders with multiple items or sellers cannot multiply other measures.


In [ ]:
items = read_query("""
SELECT
    foi.order_id,
    COUNT(*) AS item_count,
    COUNT(DISTINCT foi.product_key) AS unique_product_count,
    COUNT(DISTINCT foi.seller_key) AS seller_count,
    SUM(foi.price) AS item_revenue,
    SUM(foi.freight_value) AS freight_value,
    SUM(foi.price + foi.freight_value) AS merchandise_plus_freight,
    AVG(foi.price) AS avg_item_price,
    MAX(foi.price) AS max_item_price
FROM dw.fact_order_items AS foi
GROUP BY foi.order_id;
""")

assert items["order_id"].nunique() == len(items)
assert not items["order_id"].duplicated().any()
print("Orders with items:", len(items))


### 2.3 Payment aggregates


In [ ]:
payments = read_query("""
SELECT
    fp.order_id,
    COUNT(*) AS payment_record_count,
    COUNT(DISTINCT fp.payment_type) AS payment_type_count,
    SUM(fp.payment_value) AS payment_value,
    MAX(fp.payment_installments) AS max_installments
FROM dw.fact_payments AS fp
GROUP BY fp.order_id;
""")

assert payments["order_id"].nunique() == len(payments)
assert not payments["order_id"].duplicated().any()
print("Orders with payments:", len(payments))


### 2.4 Review aggregates

The warehouse preserves review events rather than forcing one review per order. For order-level EDA, the first and latest answered review are retained alongside event count and mean score.

`score_changed` specifically means that the first and latest review scores differ. This is not identical to asking whether any two review events differ for orders with three reviews.


In [ ]:
reviews = read_query("""
WITH ranked_reviews AS (
    SELECT
        fr.order_id,
        fr.review_score,
        fr.review_answer_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY fr.order_id
            ORDER BY fr.review_answer_timestamp ASC
        ) AS rn_first,
        ROW_NUMBER() OVER (
            PARTITION BY fr.order_id
            ORDER BY fr.review_answer_timestamp DESC
        ) AS rn_latest
    FROM dw.fact_reviews AS fr
)
SELECT
    order_id,
    COUNT(*) AS review_count,
    MAX(review_score) FILTER (WHERE rn_first = 1) AS first_review_score,
    MAX(review_score) FILTER (WHERE rn_latest = 1) AS latest_review_score,
    AVG(review_score) AS mean_review_score,
    CASE
        WHEN COUNT(*) > 1 THEN
            MAX(review_score) FILTER (WHERE rn_first = 1)
            <> MAX(review_score) FILTER (WHERE rn_latest = 1)
        ELSE FALSE
    END AS score_changed
FROM ranked_reviews
GROUP BY order_id;
""")

assert len(reviews) == 98_673
assert reviews["order_id"].nunique() == len(reviews)
assert not reviews["order_id"].duplicated().any()

print(reviews["review_count"].value_counts().sort_index())
print(reviews["score_changed"].value_counts())


### 2.5 Merge at one-row-per-order grain


In [ ]:
order_analysis = (
    orders
    .merge(items, on="order_id", how="left", validate="one_to_one")
    .merge(payments, on="order_id", how="left", validate="one_to_one")
    .merge(reviews, on="order_id", how="left", validate="one_to_one")
)

assert len(order_analysis) == 99_441
assert order_analysis["order_id"].nunique() == len(order_analysis)
assert not order_analysis["order_id"].duplicated().any()

relationship_checks = pd.Series({
    "orders_without_items": order_analysis["item_count"].isna().sum(),
    "orders_without_payments": order_analysis["payment_value"].isna().sum(),
    "orders_without_reviews": order_analysis["review_count"].isna().sum(),
})
relationship_checks


Expected relationship gaps are preserved rather than imputed: 775 orders have no item rows, one has no payment row, and 768 have no review event.


## 3. Analytical features


In [ ]:
order_analysis["purchase_year"] = order_analysis["order_purchase_timestamp"].dt.year
order_analysis["purchase_month"] = order_analysis["order_purchase_timestamp"].dt.month
order_analysis["purchase_year_month"] = order_analysis["order_purchase_timestamp"].dt.to_period("M")
order_analysis["purchase_day_of_week"] = order_analysis["order_purchase_timestamp"].dt.day_name()
order_analysis["purchase_hour"] = order_analysis["order_purchase_timestamp"].dt.hour

order_analysis["freight_ratio"] = (
    order_analysis["freight_value"]
    / order_analysis["merchandise_plus_freight"]
)

order_analysis["delivery_status"] = pd.cut(
    order_analysis["delivery_delay_days"],
    bins=[-np.inf, -7, 0, 3, 7, np.inf],
    labels=[
        "≥7d early",
        "<7d early / on time",
        "≤3d late",
        "3–7d late",
        ">7d late",
    ],
    right=True,
)

order_analysis["review_group"] = pd.cut(
    order_analysis["latest_review_score"],
    bins=[0, 2, 3, 5],
    labels=["Negative", "Neutral", "Positive"],
)

print("Shape:", order_analysis.shape)
order_analysis[["freight_ratio", "delivery_days", "delivery_delay_days"]].describe()


The delivery bins are descriptive severity groups, not source fields. Because delay is measured in fractional days, labels use inequality notation rather than implying integer-day boundaries.

Latest review scores are grouped as: 1–2 negative, 3 neutral, and 4–5 positive.


## 4. Controlled EDA population


In [ ]:
eda = order_analysis.loc[
    (order_analysis["order_status"] == "delivered")
    & (order_analysis["order_purchase_timestamp"] >= "2017-01-01")
    & (order_analysis["order_purchase_timestamp"] < "2018-09-01")
].copy()

assert eda["order_id"].nunique() == len(eda)

print("EDA rows:", len(eda))
print(
    "Period:",
    eda["order_purchase_timestamp"].min(),
    "→",
    eda["order_purchase_timestamp"].max(),
)


This reproduces the B1 comparison window. Records outside the period remain in the warehouse; the filter is analytical rather than a cleaning operation.


## 5. Marketplace evolution


In [ ]:
monthly = (
    eda.groupby("purchase_year_month", observed=True)
    .agg(
        orders=("order_id", "nunique"),
        revenue=("item_revenue", "sum"),
        freight=("freight_value", "sum"),
    )
    .reset_index()
)

monthly["aov"] = monthly["revenue"] / monthly["orders"]
monthly["order_growth_pct"] = monthly["orders"].pct_change() * 100
monthly["revenue_growth_pct"] = monthly["revenue"].pct_change() * 100
monthly["aov_growth_pct"] = monthly["aov"].pct_change() * 100

monthly


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.8))

x = monthly["purchase_year_month"].astype(str)
ax.plot(x, monthly["revenue"], marker="o", linewidth=2, markersize=4)

ax.set_title("Monthly Delivered-Order Merchandise Revenue", loc="left", fontweight="bold")
ax.set_xlabel("Purchase month")
ax.set_ylabel("Merchandise revenue")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"R${v/1_000_000:.1f}M"))
ax.grid(axis="y", alpha=0.20, linewidth=0.8)
ax.tick_params(axis="x", rotation=45)

nov = monthly.loc[monthly["purchase_year_month"].astype(str) == "2017-11"].iloc[0]
nov_idx = monthly.index[monthly["purchase_year_month"].astype(str) == "2017-11"][0]
ax.annotate(
    f"Nov 2017\nR${nov['revenue']/1_000:.0f}k",
    xy=(nov_idx, nov["revenue"]),
    xytext=(nov_idx + 1.2, nov["revenue"] * 0.91),
    arrowprops={"arrowstyle": "-", "lw": 0.8},
)

save_figure("b3_monthly_revenue.png")


**Interpretation.** November 2017 is a notable high-volume month: orders increased 62.77% month over month and merchandise revenue increased 52.37%, while AOV declined 6.39%. The pattern is therefore associated primarily with higher order volume, not higher order value. Early-2017 percentage growth should be interpreted cautiously because it starts from a much smaller activity base.


## 6. Category and seller concentration


In [ ]:
items_detail = read_query("""
SELECT
    foi.order_id,
    foi.order_item_id,
    dp.product_id,
    dp.product_category_name_english AS category,
    ds.seller_id,
    foi.price,
    foi.freight_value,
    fo.order_purchase_timestamp,
    dos.order_status
FROM dw.fact_order_items AS foi
JOIN dw.fact_orders AS fo
    ON foi.order_id = fo.order_id
JOIN dw.dim_product AS dp
    ON foi.product_key = dp.product_key
JOIN dw.dim_seller AS ds
    ON foi.seller_key = ds.seller_key
JOIN dw.dim_order_status AS dos
    ON foi.order_status_key = dos.order_status_key;
""")

assert len(items_detail) == 112_650
assert not items_detail.duplicated(["order_id", "order_item_id"]).any()

items_eda = items_detail.loc[
    (items_detail["order_status"] == "delivered")
    & (items_detail["order_purchase_timestamp"] >= "2017-01-01")
    & (items_detail["order_purchase_timestamp"] < "2018-09-01")
].copy()


In [ ]:
category_revenue = (
    items_eda.groupby("category", dropna=False)
    .agg(
        revenue=("price", "sum"),
        items=("order_item_id", "count"),
        orders=("order_id", "nunique"),
    )
    .sort_values("revenue", ascending=False)
    .reset_index()
)

category_revenue["revenue_share_pct"] = (
    category_revenue["revenue"] / category_revenue["revenue"].sum() * 100
)
category_revenue["cumulative_share_pct"] = category_revenue["revenue_share_pct"].cumsum()

category_revenue.head(15)


In [ ]:
top_categories = category_revenue.head(10).sort_values("revenue")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_categories["category"], top_categories["revenue"])

ax.set_title("Top Product Categories by Merchandise Revenue", loc="left", fontweight="bold")
ax.set_xlabel("Merchandise revenue")
ax.set_ylabel("")
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"R${v/1_000_000:.1f}M"))
ax.grid(axis="x", alpha=0.20, linewidth=0.8)

save_figure("b3_top_categories.png")


In [ ]:
seller_revenue = (
    items_eda.groupby("seller_id")
    .agg(
        revenue=("price", "sum"),
        orders=("order_id", "nunique"),
        items=("order_item_id", "count"),
    )
    .sort_values("revenue", ascending=False)
    .reset_index()
)

seller_revenue["revenue_share_pct"] = (
    seller_revenue["revenue"] / seller_revenue["revenue"].sum() * 100
)
seller_revenue["cumulative_share_pct"] = seller_revenue["revenue_share_pct"].cumsum()
seller_revenue["seller_rank"] = np.arange(1, len(seller_revenue) + 1)

for threshold in [25, 50, 75, 90]:
    n = seller_revenue["cumulative_share_pct"].ge(threshold).idxmax() + 1
    print(f"{threshold}% of revenue: {n} sellers")

print(f"Largest seller share: {seller_revenue.iloc[0]['revenue_share_pct']:.2f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.8))

ax.plot(
    seller_revenue["seller_rank"],
    seller_revenue["cumulative_share_pct"],
    linewidth=2,
)

for threshold, rank in [(50, 127), (75, 424), (90, 885)]:
    ax.axhline(threshold, linestyle="--", linewidth=0.8, alpha=0.45)
    ax.scatter(rank, threshold, s=28, zorder=3)
    ax.annotate(
        f"{threshold}%: {rank} sellers",
        xy=(rank, threshold),
        xytext=(8, -14 if threshold == 90 else 8),
        textcoords="offset points",
        fontsize=9,
    )

ax.set_title("Cumulative Merchandise Revenue Across Sellers", loc="left", fontweight="bold")
ax.set_xlabel("Seller rank by merchandise revenue")
ax.set_ylabel("Cumulative revenue share (%)")
ax.set_ylim(0, 102)
ax.grid(axis="y", alpha=0.18, linewidth=0.8)

save_figure("b3_seller_concentration.png")


The top five product categories account for 39.88% of merchandise revenue; the top 15 account for 76.33%.

Seller revenue is concentrated but not dominated by one seller. Of 2,945 active sellers, 127 (about 4.3%) account for 50% of merchandise revenue, while the largest seller contributes only 1.72%.


## 7. Delivery performance and customer satisfaction


In [ ]:
delivery_reviews = eda.loc[
    eda["delivery_delay_days"].notna()
    & eda["latest_review_score"].notna()
].copy()

delivery_summary = (
    delivery_reviews.groupby("is_late_delivery")
    .agg(
        orders=("order_id", "nunique"),
        avg_review_score=("latest_review_score", "mean"),
        median_review_score=("latest_review_score", "median"),
        avg_delivery_delay=("delivery_delay_days", "mean"),
        positive_review_rate=("latest_review_score", lambda x: (x >= 4).mean() * 100),
        negative_review_rate=("latest_review_score", lambda x: (x <= 2).mean() * 100),
    )
)

delivery_summary


In [ ]:
delivery_by_status = (
    delivery_reviews.groupby("delivery_status", observed=True)
    .agg(
        orders=("order_id", "nunique"),
        avg_review_score=("latest_review_score", "mean"),
        positive_review_rate=("latest_review_score", lambda x: (x >= 4).mean() * 100),
        negative_review_rate=("latest_review_score", lambda x: (x <= 2).mean() * 100),
    )
    .reset_index()
)

delivery_by_status


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.8))

labels = delivery_by_status["delivery_status"].astype(str)
bars = ax.bar(labels, delivery_by_status["avg_review_score"])

ax.set_title("Customer Review Score by Delivery Performance", loc="left", fontweight="bold")
ax.set_xlabel("Delivery performance relative to estimate")
ax.set_ylabel("Average latest review score")
ax.set_ylim(0, 5)
ax.grid(axis="y", alpha=0.20, linewidth=0.8)
ax.tick_params(axis="x", rotation=20)

for bar, value in zip(bars, delivery_by_status["avg_review_score"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.08,
        f"{value:.2f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

save_figure("b3_delivery_review_score.png")


In [ ]:
rho, p_value = spearmanr(
    delivery_reviews["delivery_delay_days"],
    delivery_reviews["latest_review_score"],
)

late = delivery_reviews.loc[delivery_reviews["is_late_delivery"] == True]
on_time = delivery_reviews.loc[delivery_reviews["is_late_delivery"] == False]

late_negative_rate = (late["latest_review_score"] <= 2).mean()
on_time_negative_rate = (on_time["latest_review_score"] <= 2).mean()
risk_ratio = late_negative_rate / on_time_negative_rate
risk_difference = late_negative_rate - on_time_negative_rate

print(f"Spearman rho: {rho:.3f}")
print("p-value: < 0.001" if p_value < 0.001 else f"p-value: {p_value:.3f}")
print(f"Late negative-review rate: {late_negative_rate:.2%}")
print(f"On-time/early negative-review rate: {on_time_negative_rate:.2%}")
print(f"Observed risk ratio: {risk_ratio:.2f}x")
print(f"Absolute difference: {risk_difference:.2%}")


**Interpretation.** Late deliveries have a 54.06% negative-review rate versus 9.20% for on-time/early deliveries: an observed risk ratio of 5.88× and an absolute difference of 44.86 percentage points. Review scores also decline as delay severity increases. Spearman ρ = -0.177 indicates a negative, though not large, monotonic association at the individual-order level.

These are observational associations. They do not establish that delivery lateness alone caused a customer's review score.


## 8. Geography and repeat purchasing


In [ ]:
state_summary = (
    eda.groupby("customer_state")
    .agg(
        orders=("order_id", "nunique"),
        customers=("customer_unique_id", "nunique"),
        revenue=("item_revenue", "sum"),
    )
    .sort_values("orders", ascending=False)
    .reset_index()
)

state_summary["order_share_pct"] = state_summary["orders"] / state_summary["orders"].sum() * 100
state_summary["revenue_share_pct"] = state_summary["revenue"] / state_summary["revenue"].sum() * 100

state_summary.head(10)


In [ ]:
top_states = state_summary.head(10).sort_values("order_share_pct")

fig, ax = plt.subplots(figsize=(9, 5.8))
bars = ax.barh(top_states["customer_state"], top_states["order_share_pct"])

ax.set_title("Delivered Orders by Customer State", loc="left", fontweight="bold")
ax.set_xlabel("Share of delivered orders (%)")
ax.set_ylabel("")
ax.grid(axis="x", alpha=0.20, linewidth=0.8)

for bar, value in zip(bars, top_states["order_share_pct"]):
    ax.text(
        value + 0.25,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center",
        fontsize=9,
    )

save_figure("b3_orders_by_state.png")


In [ ]:
customer_orders = (
    eda.groupby("customer_unique_id")
    .agg(
        orders=("order_id", "nunique"),
        revenue=("item_revenue", "sum"),
    )
    .reset_index()
)

customer_summary = pd.Series({
    "customers": len(customer_orders),
    "repeat_customers": (customer_orders["orders"] > 1).sum(),
    "repeat_customer_pct": (customer_orders["orders"] > 1).mean() * 100,
    "avg_orders_per_customer": customer_orders["orders"].mean(),
    "max_orders_per_customer": customer_orders["orders"].max(),
})

customer_summary


Repeat purchasing is described as the share of persistent customers with more than one delivered order during the analytical period. It is **not** labeled a retention rate because no cohort-specific eligibility or follow-up window is defined.


## 9. Conclusions

- Python reproduces the validated B1 monthly, category, seller, and delivery findings from the B2 warehouse.
- November 2017's revenue surge is associated mainly with higher order volume while AOV declined.
- Merchandise revenue is concentrated across a relatively small seller segment without dependence on a single seller.
- Delivery lateness is strongly associated with poorer customer-review outcomes, and the relationship worsens with delay severity.
- Marketplace activity is geographically concentrated, especially in São Paulo.
- Repeat purchasing is uncommon in the observed analytical window.

The next phase, B4, will translate the validated business metrics into a Power BI analytical layer and dashboard. B5 will later treat late-delivery prediction as a separate predictive problem with explicit leakage controls.
